# Ferret Colab: 下载 HF Llama 3.2 1B + Dolly 并训练 FedStructMuon

这个 notebook 会完成四件事：
1. 安装 Colab 依赖（保留 Colab 预装 PyTorch）
2. 从 Hugging Face 下载 `meta-llama/Llama-3.2-1B` 到本地目录
3. 从 Hugging Face 下载 `databricks/databricks-dolly-15k`，并转换为项目需要的 `data/databricks-dolly-15k.jsonl`
4. 运行一个可直接启动的 `fedstructmuon` 训练示例（含动态 `struct_topk` 参数）

In [ ]:
# 如果你还没把仓库放到 /content/Ferret，先取消注释并修改地址后执行：
# !git clone <YOUR_FERRET_REPO_URL> /content/Ferret
%cd /content/Ferret
!pwd

In [ ]:
%%bash
set -e
cd /content/Ferret

# Colab 已预装 torch/cuda，过滤掉 requirements.txt 里的固定 torch/nvidia 版本，避免冲突。
grep -vE '^(torch|torchaudio|torchvision|triton|nvidia-)' requirements.txt > requirements_colab.txt
pip install -q -r requirements_colab.txt

python -V

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('cuda_device_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda_0:', torch.cuda.get_device_name(0))

## 下载模型：Hugging Face Llama 3.2 1B

注意：`meta-llama/Llama-3.2-1B` 是 gated 模型。你需要先在 Hugging Face 页面同意许可证，然后在 Colab 提供 `HF_TOKEN`（建议放在 Colab Secrets 里，名字就叫 `HF_TOKEN`）。

In [ ]:
import os
from huggingface_hub import snapshot_download, login

MODEL_ID = 'meta-llama/Llama-3.2-1B'
MODEL_DIR = '/content/models/llama-3.2-1b'

hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = ''

if not hf_token:
    raise RuntimeError(
        '未检测到 HF_TOKEN。请先在 Colab Secrets 添加 HF_TOKEN，或在环境变量中设置 HF_TOKEN。'
    )

login(token=hf_token, add_to_git_credential=False)
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
    token=hf_token,
)

print('model saved to:', MODEL_DIR)

In [ ]:
from datasets import load_dataset
import os
import json

os.makedirs('data', exist_ok=True)

# HF 下载 Dolly
ds = load_dataset('databricks/databricks-dolly-15k', split='train')
out_path = 'data/databricks-dolly-15k.jsonl'

with open(out_path, 'w', encoding='utf-8') as f:
    for ex in ds:
        row = {
            'instruction': ex.get('instruction', ''),
            'context': ex.get('context', ''),
            'response': ex.get('response', ''),
            'category': ex.get('category', ''),
        }
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('saved:', out_path)
print('rows :', len(ds))

In [ ]:
import json

with open('data/databricks-dolly-15k.jsonl', 'r', encoding='utf-8') as f:
    for i in range(2):
        item = json.loads(next(f))
        print(f'example[{i}] keys:', list(item.keys()))
        print('instruction:', item['instruction'][:80])
        print('response   :', item['response'][:80])
        print('-' * 60)

## 训练（FedStructMuon 示例）

说明：
- 这是 smoke 配置，优先保证能在 Colab 先跑通。
- 训练默认使用上面下载到本地的 `/content/models/llama-3.2-1b`。
- `--dataset_subsample 0.1` 只取 10% 数据，方便快速测试。

In [ ]:
%%bash
set -e
cd /content/Ferret

MODEL_DIR=/content/models/llama-3.2-1b

python main.py \
  --algo fedstructmuon \
  --dataset dolly \
  --model ${MODEL_DIR} \
  --rounds 2 \
  --local_step 20 \
  --num_clients 20 \
  -m 0.1 \
  --batch_size 1 \
  --max_length 512 \
  --dataset_subsample 0.1 \
  --device 0 \
  --log \
  --save \
  --equal_weight \
  --rank_r 8 \
  --struct_num_subspaces 4 \
  --struct_topk 20 \
  --struct_topk_init_warmup 1 \
  --struct_topk_final_warmup 2 \
  --struct_topk_tt 1.0

In [ ]:
import glob
import os

log_dirs = sorted(glob.glob('logs/*'), key=os.path.getmtime)
latest = log_dirs[-1] if log_dirs else None
print('latest log dir:', latest)

if latest is not None:
    best_eval_ready = os.path.join(latest, 'best_eval_ready.pt')
    best_pt = os.path.join(latest, 'best.pt')
    print('best_eval_ready.pt exists:', os.path.exists(best_eval_ready))
    print('best.pt exists          :', os.path.exists(best_pt))

如需单独评估：

```bash
python evaluate_dolly.py \
  --model /content/models/llama-3.2-1b \
  --checkpoint <logs/.../best_eval_ready.pt 或 best.pt> \
  --dataset dolly \
  --device 0 \
  --eval_metric rouge
```